# DAEN 328 Assignment 5: Load and Serve

**Name:** _your name here_

Save this notebook as `A5_LastName_FirstName.ipynb`. It must run **top to bottom** (Kernel > Restart & Run All) with all outputs showing when you submit.

Work through `Assignment05_Guide.ipynb` first. Every technique you need is taught there on practice data. This notebook is where you apply it to `EmployeeDB`.

Your dashboard (Part C) is a separate file, `app.py`. See the instructions PDF.

## CONFIG: edit this cell once

In [2]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine, text

DB_NAME = "EmployeeDB"
DB_USER = "postgres"
DB_PASSWORD = "admin"      # change to YOUR PostgreSQL password
DB_HOST = "localhost"
DB_PORT = "5432"

CSV_FILE = "Cleaned_data.csv"   # your Assignment 3 output, in the same folder as this notebook

DB_PARAMS = {"dbname": DB_NAME, "user": DB_USER, "password": DB_PASSWORD,
             "host": DB_HOST, "port": DB_PORT}
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

ModuleNotFoundError: No module named 'psycopg2'

# Part A: Load your cleaned CSV into the 7 EmployeeDB tables

### A1. Read and inspect

Read the CSV with every column as text. Show its shape and the number of blanks per column.

In [100]:
# TODO A1
#Read CSV with every column as text Show its shape and the number of blanks per column.
df = pd.read_csv("data.csv", dtype=str)
print("Shape of the DataFrame:", df.shape)
print("Number of blanks per column:")
print(df.isnull().sum())

FileNotFoundError: [Errno 2] No such file or directory: 'data.csv'

### A2. Clean and validate in pandas

Required:
- Strip spaces from text columns.
- Convert `Birthdate` to a real date (`pd.to_datetime(..., errors="coerce").dt.date`) and **print how many could not be read**.
- Convert `Salary` and `EmployeeID` to numbers.
- Put every row that cannot satisfy your schema's NOT NULL or CHECK constraints into a `rejects` DataFrame, save it as `rejects_employees.csv`, and print rows read / good / rejected. Never invent a value (such as a salary of 0) to get a row past a constraint.

In [ ]:
# TODO A2

### Given: `clean()` (run it, do not change it)

Pass **every** value through `clean()` when you insert it. pandas marks a blank as `NaN`, and if a `NaN` reaches PostgreSQL it is stored as the value `NaN`, not as NULL. A `NaN` salary passes `NOT NULL` and `CHECK (salary > 0)` with no error, and then every average that includes it comes out as `NaN`. `clean()` turns blanks into `None` (a real NULL), so the database rejects what it should. It also turns numpy numbers into plain Python numbers, which avoids the error `can't adapt type numpy.int64`.

In [ ]:
def clean(value):
    # Turn pandas blanks into None (SQL NULL) and numpy numbers into plain Python numbers.
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value

### Given: start over with empty tables

While you build the loader, a bad attempt can leave wrong rows behind. To empty all 7 tables and restart the id numbers from 1, set `RESET = True`, run the cell, then set it back to `False`. This keeps your tables and constraints; it only removes the rows. Do not use `DROP TABLE`, which deletes your Assignment 4 schema.

Leave `RESET = False` when you submit.

In [ ]:
RESET = False

if RESET:
    with engine.begin() as conn:
        conn.execute(text("""
            TRUNCATE employeedetails, personaldetails, jobdescriptions, divisionlocations,
                     departments, jobtitles, divisions
            RESTART IDENTITY CASCADE;
        """))
    print("All 7 tables emptied.")

### A3. The loader

Write `load_employees(good)`. Requirements:
- Loads all 7 tables, lookup tables first, `employeedetails` last, using the CSV-to-column map in the instructions PDF.
- Runs in **one transaction** (all or nothing).
- **Safe to re-run**: a second run adds zero rows to every table.
- Uses `%s` placeholders for every value, and passes every value through `clean()`.

In [ ]:
# TODO A3
def load_employees(good):
    ...

### A4. Run it twice and reconcile

Run your loader, record the 7 table counts, run it **again**, record them again. Show one table with, for every table: the count you expected from the CSV, the count after run 1, the count after run 2, and whether all three agree.

Then run the given NaN check below it. Every count must be 0.

In [ ]:
# TODO A4

In [ ]:
# Given: NaN check. Every count must be 0. If not, a value reached the database without clean().
with engine.connect() as conn:
    nan_check = {
        "salary = NaN":    conn.execute(text("SELECT COUNT(*) FROM employeedetails WHERE salary = 'NaN';")).scalar(),
        "name = 'NaN'":    conn.execute(text("SELECT COUNT(*) FROM personaldetails WHERE name = 'NaN';")).scalar(),
        "jobtitle = 'NaN'": conn.execute(text("SELECT COUNT(*) FROM jobtitles WHERE jobtitle = 'NaN';")).scalar(),
    }
nan_check

### A5. Load report (write in this cell, 4 to 6 sentences)

- How many rows you read, loaded and rejected, and the reason for each kind of reject.
- Which loading pattern you used for `divisionlocations`, `jobdescriptions` and `employeedetails`, and why.
- One real problem you hit while building the loader (the error message or wrong count) and how you found and fixed it.


_Your answer here._

# Part B: SQL

Write your own SQL for each question in the code cell under it, using the `q()` helper, and leave the result showing. Rules for every question:
- Filter by **names** (`'Engineering'`, `'Austin'`), never by an id number you looked up.
- Never type a number that the database can compute (an average, a count).
- Show names, not ids, wherever a person would read the result.

In [ ]:
def q(sql, params=None):
    return pd.read_sql(text(sql), engine, params=params)

**Q1.** Headcount by department: each department's name and number of employees, largest first.

In [ ]:
# Q1
q('''

''')

**Q2.** Engineering employees earning less than $100,000: name, job title and salary, highest salary first.

In [ ]:
# Q2
q('''

''')

**Q3.** Human Resources employees who work at the division located in **Austin**: name, job title and division. Find the division through its city; do not type the division's name.

In [ ]:
# Q3
q('''

''')

**Q4.** Human Resources only: for each division, the number of HR employees and their average salary (2 decimals), highest average first.

In [ ]:
# Q4
q('''

''')

**Q5.** Engineering employees whose salary is at or above the average Engineering salary: name, job title, salary. The average must be computed inside the query.

In [ ]:
# Q5
q('''

''')

**Q6.** Employees aged 50 or older versus under 50 (age as of today): for each group, the number of employees and the average salary. Leave out anyone with no birthdate.

In [ ]:
# Q6
q('''

''')

**Q7.** For each division: its city and state, its headcount and its average salary, largest headcount first.

In [ ]:
# Q7
q('''

''')

**Q8.** Departments whose average salary is higher than the company-wide average salary, with that department average.

In [ ]:
# Q8
q('''

''')

**Q9.** The highest-paid employee in each department: department, name, job title and salary.

In [ ]:
# Q9
q('''

''')

**Q10.** Data quality: every email address that belongs to more than one person, with how many people share it. Then answer in the markdown cell below.

In [ ]:
# Q10
q('''

''')

**Q10, part 2 (2 to 3 sentences):** Look at the rows Q10 returned. Is a shared email a data error, or could it be legitimate? KLB wants to use email as each employee's login. What would you check or change in the data or the schema before that is safe?


_Your answer here._

# Part C: Dashboard

Your dashboard is `app.py`, a separate file. Build it from the specification in the instructions PDF, run it with `streamlit run app.py`, and put your screenshots and answers in the PDF. Nothing for Part C goes in this notebook.

# Part D: Industry scenarios

These are the kind of incidents data engineers handle at large companies. There is no single right answer. Answer each one in the same cell, under the question, in 3 to 5 sentences. Refer to **your own** loader, SQL or `app.py` wherever you can: a specific answer (the line, the query, the setting) beats a general one.

**D1. The job said SUCCESS.** Your loader runs every night. For months it has rejected a handful of rows each night. One Monday it rejects about 4,100 of 14,000 rows, the job still reports SUCCESS, and the dashboard's headcount drops by almost a third. Nobody notices for two days. The cause: the HR system started sending salaries as `"$85,000.00"` instead of `85000`.

1. Look at how **your** loader converts `Salary`. What would it do with `"$85,000.00"`, and where would those rows end up? If your loader would behave differently from the story above, say how.
2. Why did nothing fail?
3. What would you add to the loader so that next time this stops the run or alerts someone?


_Your answer here._

**D2. Two numbers for the same thing.** Finance's report and your dashboard both read the same `EmployeeDB`, but they show different average salaries for Engineering. Your manager asks which one is wrong.

1. Give three legitimate reasons the two numbers could differ without either query having a bug. At least one reason must come from how your own `app.py` works.
2. How would you settle which number the company should use?


_Your answer here._

**D3. From 14,000 rows to 50 million.** After KLB acquires another company, the nightly file grows from about 14,000 rows to 50 million.

1. Time one run of your loader on your own data (for example, wrap one of the runs in A4 with `time.perf_counter()`) and report the time.
2. What in your loader would break or slow down first at 50 million rows? Point to the specific part of your code and explain why it gets slow.
3. What would you change?


_Your answer here._